# HBR YouTube OCR

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/Data_crawling/blob/main/hbr_youtube_ocr/hbr_youtube_ocr_colab.ipynb?skip_cache=true)

YouTube 영상 다운로드 → 전체 WAV 추출 → OpenCV 고정 ROI → MangaOCR → 화자/대사 timestamp 매칭.

Colab 링크는 `skip_cache=true`를 사용해 GitHub의 최신 노트북을 다시 불러옵니다.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install -U pytubefix manga-ocr opencv-python-headless
!pip -q install pandas==2.2.3
!pip -q install --force-reinstall --no-cache-dir pillow==11.3.0


In [ ]:
SOURCE_MODE = "youtube" # @param ["youtube", "upload", "gdrive"]
YOUTUBE_URL = "https://www.youtube.com/watch?v=zHN25OVshIY" # @param {type:"string"}
GDRIVE_VIDEO_PATH = "/content/drive/MyDrive/video.mp4" # @param {type:"string"}
OUTPUT_DIR = "/content/hbr_youtube_ocr" # @param {type:"string"}

from pathlib import Path
import subprocess
import time

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
video_path = output_dir / "source.mp4"
audio_path = output_dir / "full_audio.wav"

def stage(message):
    print(f"\n[STEP] {message}", flush=True)

if SOURCE_MODE == "youtube":
    from pytubefix import YouTube

    download_state = {"last_percent": -1}

    def on_progress(stream, chunk, bytes_remaining):
        total = stream.filesize or stream.filesize_approx
        if not total:
            return
        downloaded = total - bytes_remaining
        percent = int(downloaded * 100 / total)
        if percent != download_state["last_percent"] and (percent % 5 == 0 or percent == 100):
            print(
                f"    download {percent:3d}%  "
                f"{downloaded / 1024 / 1024:.1f}/{total / 1024 / 1024:.1f} MB",
                flush=True,
            )
            download_state["last_percent"] = percent

    stage("1/6 YouTube metadata 읽는 중")
    t0 = time.time()
    yt = YouTube(YOUTUBE_URL, on_progress_callback=on_progress)
    print("title:", yt.title)
    print(f"metadata done: {time.time() - t0:.1f}s")

    stage("2/6 다운로드 가능한 스트림 조회 중")
    t0 = time.time()
    video_stream = (
        yt.streams
        .filter(adaptive=True, only_video=True, file_extension="mp4")
        .order_by("resolution")
        .desc()
        .first()
    )
    audio_stream = (
        yt.streams
        .filter(adaptive=True, only_audio=True)
        .order_by("abr")
        .desc()
        .first()
    )

    if video_stream is None:
        raise RuntimeError("adaptive video stream not found")
    if audio_stream is None:
        raise RuntimeError("adaptive audio stream not found")

    print("video:", video_stream.resolution, video_stream.mime_type, video_stream.itag)
    print("audio:", audio_stream.abr, audio_stream.mime_type, audio_stream.itag)
    print(f"stream lookup done: {time.time() - t0:.1f}s")

    stage("3/6 비디오 스트림 다운로드 중")
    download_state["last_percent"] = -1
    t0 = time.time()
    video_only_path = Path(video_stream.download(
        output_path=str(output_dir),
        filename="video_only.mp4",
    ))
    print(f"video download done: {time.time() - t0:.1f}s")

    stage("4/6 오디오 스트림 다운로드 중")
    download_state["last_percent"] = -1
    t0 = time.time()
    audio_only_path = Path(audio_stream.download(
        output_path=str(output_dir),
        filename="audio_only.m4a",
    ))
    print(f"audio download done: {time.time() - t0:.1f}s")

    stage("5/6 비디오 + 오디오 병합 중 (재인코딩 없음)")
    t0 = time.time()
    subprocess.run([
        "ffmpeg", "-y", "-stats",
        "-i", str(video_only_path),
        "-i", str(audio_only_path),
        "-c:v", "copy",
        "-c:a", "copy",
        str(video_path),
    ], check=True)
    print(f"merge done: {time.time() - t0:.1f}s")

elif SOURCE_MODE == "upload":
    from google.colab import files

    stage("로컬 영상 업로드")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("업로드된 영상이 없습니다.")
    uploaded_name = next(iter(uploaded))
    uploaded_path = Path(uploaded_name)
    subprocess.run(["cp", str(uploaded_path), str(video_path)], check=True)
    print("video:", video_path)

elif SOURCE_MODE == "gdrive":
    from google.colab import drive

    stage("Google Drive 마운트")
    drive.mount("/content/drive")
    source_path = Path(GDRIVE_VIDEO_PATH)
    if not source_path.exists():
        raise FileNotFoundError(source_path)
    video_path = source_path
    print("video:", video_path)

else:
    raise ValueError(f"unknown SOURCE_MODE: {SOURCE_MODE}")

stage("전체 WAV 추출 중")
t0 = time.time()
subprocess.run([
    "ffmpeg", "-y", "-stats",
    "-i", str(video_path),
    "-vn",
    "-c:a", "pcm_s16le",
    str(audio_path),
], check=True)
print(f"wav extraction done: {time.time() - t0:.1f}s")

print("\n[DONE]")
print("video:", video_path)
print("audio:", audio_path)


In [ ]:
PREVIEW_TIME_SEC = 60.0 # @param {type:"number"}
SPEAKER_X1 = 0.050 # @param {type:"number"}
SPEAKER_Y1 = 0.600 # @param {type:"number"}
SPEAKER_X2 = 0.190 # @param {type:"number"}
SPEAKER_Y2 = 0.695 # @param {type:"number"}
DIALOGUE_X1 = 0.155 # @param {type:"number"}
DIALOGUE_Y1 = 0.710 # @param {type:"number"}
DIALOGUE_X2 = 0.900 # @param {type:"number"}
DIALOGUE_Y2 = 0.820 # @param {type:"number"}

import cv2
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(str(video_path))
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration = cap.get(cv2.CAP_PROP_FRAME_COUNT) / fps
cap.set(cv2.CAP_PROP_POS_MSEC, PREVIEW_TIME_SEC * 1000)
ok, frame = cap.read()
cap.release()
if not ok:
    raise RuntimeError("preview frame read failed")

def to_box(x1, y1, x2, y2):
    return int(x1*width), int(y1*height), int(x2*width), int(y2*height)

speaker_box = to_box(SPEAKER_X1,SPEAKER_Y1,SPEAKER_X2,SPEAKER_Y2)
dialogue_box = to_box(DIALOGUE_X1,DIALOGUE_Y1,DIALOGUE_X2,DIALOGUE_Y2)

preview = frame.copy()
cv2.rectangle(preview, speaker_box[:2], speaker_box[2:], (0,255,0), 3)
cv2.rectangle(preview, dialogue_box[:2], dialogue_box[2:], (255,0,0), 3)
print(f"{width}x{height}, fps={fps:.3f}, duration={duration:.1f}s")
plt.figure(figsize=(16,9))
plt.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()


In [ ]:
from manga_ocr import MangaOcr
from PIL import Image
import re

mocr = MangaOcr()

def crop_roi(img, box):
    x1,y1,x2,y2 = box
    return img[y1:y2, x1:x2]

def prep(crop):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, 170, 255, cv2.THRESH_BINARY)
    return Image.fromarray(cv2.bitwise_not(binary)).convert("RGB")

def norm(text):
    return re.sub(r"\s+", "", text).strip()

print("speaker:", norm(mocr(prep(crop_roi(frame, speaker_box)))))
print("dialogue:", norm(mocr(prep(crop_roi(frame, dialogue_box)))))


In [ ]:
SAMPLE_INTERVAL_SEC = 0.5 # @param {type:"number"}
PREVIEW_EVERY_N = 100 # @param {type:"integer"}

from difflib import SequenceMatcher
from IPython.display import clear_output, display
from tqdm.auto import tqdm
import math
import pandas as pd

def similar(a, b):
    if not a or not b:
        return False
    return a.startswith(b) or b.startswith(a) or SequenceMatcher(None,a,b).ratio() >= 0.82

cap = cv2.VideoCapture(str(video_path))
duration = cap.get(cv2.CAP_PROP_FRAME_COUNT) / cap.get(cv2.CAP_PROP_FPS)
total_samples = math.floor(duration / SAMPLE_INTERVAL_SEC) + 1
rows = []

progress = tqdm(total=total_samples, desc="OCR", unit="sample")

for sample_index in range(total_samples):
    t = sample_index * SAMPLE_INTERVAL_SEC
    cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
    ok, img = cap.read()
    if not ok:
        break

    speaker = norm(mocr(prep(crop_roi(img, speaker_box))))
    dialogue = norm(mocr(prep(crop_roi(img, dialogue_box))))

    rows.append({
        "time_sec": round(t, 3),
        "speaker": speaker,
        "dialogue": dialogue,
    })

    progress.update(1)
    progress.set_postfix({
        "time": f"{t:.1f}/{duration:.1f}s",
        "speaker": speaker[:12],
    })

    if PREVIEW_EVERY_N > 0 and (sample_index % PREVIEW_EVERY_N == 0 or sample_index == total_samples - 1):
        clear_output(wait=True)
        display(progress.container)
        print(f"sample: {sample_index + 1}/{total_samples}")
        print(f"time: {t:.1f}/{duration:.1f} sec")
        print("speaker:", speaker)
        print("dialogue:", dialogue)

        debug = img.copy()
        cv2.rectangle(debug, speaker_box[:2], speaker_box[2:], (0, 255, 0), 3)
        cv2.rectangle(debug, dialogue_box[:2], dialogue_box[2:], (255, 0, 0), 3)

        plt.figure(figsize=(14, 8))
        plt.imshow(cv2.cvtColor(debug, cv2.COLOR_BGR2RGB))
        plt.axis("off")
        plt.show()

cap.release()
progress.close()

raw_df = pd.DataFrame(rows)
raw_df.to_csv(output_dir / "ocr_raw_samples.csv", index=False, encoding="utf-8-sig")

segments = []
current = None
for row in rows:
    s, d, ts = row["speaker"], row["dialogue"], row["time_sec"]
    if not d:
        if current is not None:
            current["end_sec"] = ts
            segments.append(current)
            current = None
        continue

    if current is None:
        current = {
            "start_sec": ts,
            "end_sec": ts,
            "speaker": s,
            "dialogue": d,
        }
        continue

    if (s == current["speaker"] or not s) and similar(d, current["dialogue"]):
        current["end_sec"] = ts
        if len(d) > len(current["dialogue"]):
            current["dialogue"] = d
        if not current["speaker"] and s:
            current["speaker"] = s
    else:
        segments.append(current)
        current = {
            "start_sec": ts,
            "end_sec": ts,
            "speaker": s,
            "dialogue": d,
        }

if current is not None:
    segments.append(current)

segments_df = pd.DataFrame(segments)
segments_df.to_csv(output_dir / "dialogue_segments.csv", index=False, encoding="utf-8-sig")

print(f"done: {len(rows)}/{total_samples} samples")
display(segments_df.head(50))
